# Pipeline GPU n°1 : PyTorch → ONNX → TensorRT → CUDA/cuDNN
TP1 Déploiement IA — ResNet-18 / CIFAR-10. **Environnement d'exécution Colab : GPU T4 (Exécution > Modifier le type d'exécution > T4 GPU).**

| Niveau | Composant |
|---|---|
| Haut niveau | PyTorch (ResNet-18 pré-entraîné) |
| Format | ONNX (opset 17) |
| Middle level | TensorRT : fusion de couches, choix des tactiques (kernels), FP32 / FP16 |
| Bas niveau | CUDA, cuDNN, cuBLAS et kernels propres à TensorRT |
| Matériel | GPU Google Colab (T4) |

[Ouvrir dans Colab](https://colab.research.google.com/github/Romain-LM-Dev/TP1_deploiement_IA/blob/main/notebooks/03_gpu_tensorrt.ipynb)

## Protocole (identique aux 4 pipelines, défini dans `common.py`)
- **Données** : les 50 premières images du jeu de test CIFAR-10, redimensionnées de 32×32 à 224×224 (bilinéaire) puis normalisées avec la moyenne/écart-type ImageNet.
- **Modèle** : ResNet-18 pré-entraîné ImageNet (`IMAGENET1K_V1`). Les 1000 classes sont regroupées en 10 classes CIFAR (ex. les 118 races de chiens → *dog*) ; la classe prédite est celle dont le meilleur logit est le plus élevé.
- **Mesure** : batchs de 10 images, données déjà sur le périphérique, 1 appel initial (compilation éventuelle, chronométré à part), 3 passes de chauffe, puis 10 exécutions mesurées des 50 images. Synchronisation explicite après chaque batch sur GPU.
- **Énergie** : compteur matériel (NVML sur GPU, RAPL sur CPU si accessible), intégré sur ≥ 3 s d'inférence en boucle après chaque exécution.
- **Référence** : PyTorch *eager* sur le même matériel, pour mesurer le gain et vérifier que les prédictions ne sont pas dégradées.

In [ ]:
!pip install -q tensorrt onnx nvidia-ml-py
!nvidia-smi

In [ ]:
# Récupération du dépôt : code commun (données, modèle, protocole de mesure)
import os, subprocess, sys
REPO = "https://github.com/Romain-LM-Dev/TP1_deploiement_IA.git"
if os.path.exists("../common.py"):          # exécution locale depuis notebooks/
    os.chdir("..")
elif not os.path.exists("common.py"):       # Google Colab
    if not os.path.exists("TP1_deploiement_IA"):
        subprocess.run(["git", "clone", "-q", REPO], check=True)
    os.chdir("TP1_deploiement_IA")
sys.path.insert(0, os.getcwd())
import common as C
C.env_info()

In [ ]:
import time
from collections import Counter
import numpy as np, torch
from IPython.display import display

torch.manual_seed(C.SEED)
x, y = C.load_data()            # (50, 3, 224, 224), normalisation ImageNet
model = C.load_model()          # ResNet-18 IMAGENET1K_V1, mode eval
print("Entrée :", tuple(x.shape), "| batch :", C.BATCH_SIZE, "| répétitions :", C.N_RUNS)
print("Répartition des classes :", dict(zip(C.CIFAR_CLASSES, np.bincount(y, minlength=10).tolist())))
results = []

In [ ]:
assert torch.cuda.is_available(), "Activer un GPU : Exécution > Modifier le type d'exécution"
torch.backends.cudnn.benchmark = True
meter = C.EnergyMeter("cuda")
print("Source énergie :", meter.source)
onnx_path = C.export_onnx(model)        # export depuis le modèle CPU, avant le passage sur GPU
model = model.cuda()
batches = C.make_batches(x.cuda())     # données déjà sur le GPU : on mesure l'inférence seule
sync = torch.cuda.synchronize

## 1. Référence : PyTorch eager sur GPU

In [ ]:
with torch.inference_mode():
    eager = C.benchmark("GPU-1 TensorRT", "PyTorch eager (réf.)", model, batches, y, device="cuda",
                        ref=None, sync=sync, meter=meter)
ref = eager["_logits"]
results.append(eager)

## 2. Construction des moteurs TensorRT (middle level)
TensorRT importe le graphe ONNX, fusionne les couches (Conv+BN+ReLU+Add en un seul kernel), puis chronomètre plusieurs implémentations (« tactiques ») de chaque couche sur le GPU pour garder la plus rapide. On construit un moteur FP32 (comparaison équitable) et un moteur FP16 (Tensor Cores du T4).

In [ ]:
import json, onnx, tensorrt as trt
print("TensorRT", trt.__version__)
LOGGER = trt.Logger(trt.Logger.WARNING)
n_onnx = len(onnx.load(onnx_path).graph.node)

def build_engine(fp16):
    builder = trt.Builder(LOGGER)
    network = builder.create_network(0)          # TensorRT 10 : batch explicite par défaut
    parser = trt.OnnxParser(network, LOGGER)
    with open(onnx_path, "rb") as f:
        if not parser.parse(f.read()):
            raise RuntimeError([str(parser.get_error(i)) for i in range(parser.num_errors)])
    config = builder.create_builder_config()
    config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
    config.profiling_verbosity = trt.ProfilingVerbosity.DETAILED
    if fp16:
        config.set_flag(trt.BuilderFlag.FP16)
    t0 = time.perf_counter()
    plan = builder.build_serialized_network(network, config)
    build_s = time.perf_counter() - t0
    engine = trt.Runtime(LOGGER).deserialize_cuda_engine(plan)
    info = json.loads(engine.create_engine_inspector().get_engine_information(trt.LayerInformationFormat.JSON))
    layers = [l for l in info["Layers"] if isinstance(l, dict)]
    print(f"{'FP16' if fp16 else 'FP32'} : construit en {build_s:.1f} s | {n_onnx} nœuds ONNX -> "
          f"{network.num_layers} couches TensorRT -> {engine.num_layers} couches dans le moteur")
    for l in layers[:4]:
        print("   ", l.get("Name", "")[:90], "|", l.get("TacticName", ""))
    return engine, build_s

class TRTRunner:
    """Exécute le moteur sur le flux CUDA de PyTorch, sans copie : les tenseurs torch servent de buffers."""
    def __init__(self, engine):
        self.ctx = engine.create_execution_context()
    def __call__(self, b):
        out = torch.empty((b.shape[0], 1000), device="cuda")
        self.ctx.set_tensor_address("input", b.data_ptr())
        self.ctx.set_tensor_address("logits", out.data_ptr())
        self.ctx.execute_async_v3(torch.cuda.current_stream().cuda_stream)
        return out

## 3. Mesures TensorRT FP32 et FP16

In [ ]:
for fp16 in (False, True):
    engine, build_s = build_engine(fp16)
    r = C.benchmark("GPU-1 TensorRT", f"TensorRT {'FP16' if fp16 else 'FP32'}", TRTRunner(engine), batches, y,
                    device="cuda", ref=ref, sync=sync, meter=meter)
    r["compile_s"] = build_s
    results.append(r)

## 4. Synthèse, graphiques et sauvegarde

In [ ]:
df = C.plot_summary(results, "Pipeline GPU n°1 : PyTorch → ONNX → TensorRT → CUDA/cuDNN", "figures/03_gpu_tensorrt.png")
display(df[C.TABLE_COLS].round(4).T)
path = C.save_results("03_gpu_tensorrt", results, {"tensorrt": trt.__version__})

In [ ]:
# Colab dans le navigateur : télécharge le JSON (à déposer dans results/ du dépôt).
# Extension Colab de VS Code : rien à faire, il suffit d'enregistrer le notebook (résultats intégrés ci-dessus).
try:
    from google.colab import files
    files.download(path)
except Exception:
    pass

## Lecture des résultats
- **Débit / temps** : comparer chaque variante à la référence *eager* de ce notebook (même machine, même session).
- **Stabilité** : un coefficient de variation (`throughput_cv_pct`) < 5 % indique des mesures reproductibles ; au-delà, regarder le *boxplot* (valeurs aberrantes = interférences de la VM partagée).
- **Qualité** : `agreement_vs_eager` doit valoir 100 % et `max_abs_diff_vs_eager` rester faible (~1e-4 en FP32).
- L'analyse croisée des 4 pipelines est faite dans `05_comparaison.ipynb` et dans le rapport.